In [1]:
# xarray to read NETCDF
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import dask as dd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'darkgrey'

In [2]:
sst = xr.open_dataset('data/SST/sst.mnmean.nc')
sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby(['lon'])
sst_df = sst.to_dataframe().reset_index()
sst_df = sst_df.query('time_bnds > 0 and nbnds == 0')
sst_df['month'] = sst_df['time'].dt.month
sst_df['year'] = sst_df['time'].dt.year
sst_df = sst_df.query('year >= 1993 and year <= 2024').reset_index().drop(['index'], axis=1)

In [3]:
chirps = xr.open_dataset('data/CHIRPS/chirps-v2.0.monthly.nc')

In [4]:
ltm_sst = sst_df.dropna().groupby(['lat', 'lon', 'month']).mean('sst').reset_index()
ltm_sst['std'] = sst_df.dropna().groupby(['lat', 'lon', 'month']).std().reset_index()['sst']
ltm_sst_clean = ltm_sst.drop(['time_bnds', 'nbnds', 'year'], axis=1)
ltm_sst_clean = ltm_sst_clean.rename(columns={'sst':'ltm_sst'})

In [5]:
sst_anomaly = sst_df.drop(['time_bnds', 'nbnds'], axis=1).merge(ltm_sst_clean, on=['lat', 'lon', 'month'], how='left')
sst_anomaly['sst_anomaly'] = sst_anomaly['sst'] - sst_anomaly['ltm_sst']
sst_anomaly['normalized_sst_anomaly'] = sst_anomaly['sst_anomaly'] / sst_anomaly['std']

In [15]:
chirps_eastern_east_africa = chirps.sel(time=slice('1993-01-01', '2024-01-01'), latitude=slice(-3.5, 8), longitude=slice(38, 50)).to_dataframe().reset_index()
chirps_eastern_east_africa['month'] = chirps_eastern_east_africa['time'].dt.month
chirps_eastern_east_africa['year'] = chirps_eastern_east_africa['time'].dt.year

month_to_season = {
    3: 'MAM', 4: 'MAM', 5: 'MAM',   # March, April, May
}

season = ['MAM']

chirps_eastern_east_africa['season'] = chirps_eastern_east_africa['month'].map(month_to_season)

chirps_eastern_east_africa_season = chirps_eastern_east_africa.dropna(subset=['season']).groupby(['year', 'season']).mean().reset_index()

In [16]:
chirps_eastern_east_africa_season

,year,season,latitude,longitude,time,precip,month
0,1993,MAM,2.249999,43.999996,1993-03-31 16:00:00.000000000,63.102024,4.0
1,1994,MAM,2.249999,43.999996,1994-03-31 16:00:00.000000000,70.791168,4.0
2,1995,MAM,2.249999,43.999996,1995-03-31 16:00:00.000000000,75.360275,4.0
3,1996,MAM,2.249999,43.999996,1996-03-31 16:00:00.000000000,66.239967,4.0
4,1997,MAM,2.249999,43.999996,1997-03-31 16:00:00.000000000,81.642693,4.0
5,1998,MAM,2.249999,43.999996,1998-03-31 16:00:00.000000000,73.752174,4.0
6,1999,MAM,2.249999,43.999996,1999-03-31 15:59:59.999999872,49.764690,4.0
7,2000,MAM,2.249999,43.999996,2000-03-31 16:00:00.000000000,48.286407,4.0
8,2001,MAM,2.249999,43.999996,2001-03-31 16:00:00.000000128,55.360474,4.0
9,2002,MAM,2.249999,43.999996,2002-03-31 16:00:00.000000000,66.756470,4.0


In [6]:
chirps_eastern_east_africa = chirps.sel(latitude=slice(-3.5, 8), longitude=slice(38, 50)).to_dataframe().reset_index()
chirps_eastern_east_africa['month'] = chirps_eastern_east_africa['time'].dt.month
chirps_eastern_east_africa['year'] = chirps_eastern_east_africa['time'].dt.year

month_to_season = {
    3: 'MAM', 4: 'MAM', 5: 'MAM',   # March, April, May
    10: 'OND', 11: 'OND', 12: 'OND' # October, November, December
}

season = ['MAM', 'OND']

chirps_eastern_east_africa['season'] = chirps_eastern_east_africa['month'].map(month_to_season)

sst_all = {}

for i in range(len(season)):
    chirps_eastern_east_africa_season = chirps_eastern_east_africa.dropna(subset=['season']).query(f'season == "{season[i]}"').groupby(['season', 'year']).mean('precip').reset_index()

    chirps_eastern_east_africa_season = chirps_eastern_east_africa_season.query('year >= 1993 and year <= 2024').drop(['month', 'latitude', 'longitude'], axis=1)

    # Get tercile values
    tercile_list = chirps_eastern_east_africa_season.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()

    season_bn = chirps_eastern_east_africa_season.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    season_n = chirps_eastern_east_africa_season.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
    season_an = chirps_eastern_east_africa_season.query(f'{tercile_list[1]} <= precip')['year'].to_list()

    sst_anomaly_season = sst_anomaly.query(f'year >= 1993 and year <= 2024')
    sst_anomaly_season['season'] = sst_anomaly_season['month'].map(month_to_season)
    sst_anomaly_season = sst_anomaly_season.query(f'season == "{season[i]}"')

    sst_df_season_bn = sst_anomaly_season[sst_anomaly_season['year'].isin(season_bn)]
    sst_df_season_n = sst_anomaly_season[sst_anomaly_season['year'].isin(season_n)]
    sst_df_season_an = sst_anomaly_season[sst_anomaly_season['year'].isin(season_an)]

    sst_df_season_dict = {'bn':sst_df_season_bn, 'n':sst_df_season_n, 'an':sst_df_season_an}

    sst_df_season = pd.concat(sst_df_season_dict.values(), keys=sst_df_season_dict.keys(), names=['tercile']).reset_index().drop(['level_1'], axis=1)

    sst_all[i] = sst_df_season.dropna().groupby(['tercile', 'lat', 'lon', 'season']).mean().reset_index()

In [7]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon', 'season', 'tercile']).to_xarray()['sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='season',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.suptitle('Eastern East Africa')
plt.savefig(f'figures/global_sst/sst_eea_season.png')
plt.close()

In [8]:
pd.concat(sst_all.values()).drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon', 'season', 'tercile']).to_xarray()['normalized_sst_anomaly'].plot(
    subplot_kws=dict(projection=ccrs.PlateCarree()),
    transform=ccrs.PlateCarree(),
    col='season',
    row='tercile',
    vmin=-1,
    vmax=1,
    cmap='RdBu_r'
    )
plt.suptitle('Eastern East Africa')
plt.savefig('figures/global_sst/normalized_sst_eea_season.png')
plt.close()